# Autoencoder and Variational Autoencoder (VAE) on Fashion-MNIST (PyTorch)

**Day 9 of 10 · Deep Learning Architectures Series**

The first **generative** project of the series. Until now every model mapped an input to a label. Here the goal is to learn the **distribution of the images themselves**, so that the model can compress, reconstruct and **generate** new images.

I compare a plain **autoencoder** with a **variational autoencoder (VAE)** and a family of **β-VAEs**, and I look at them from three angles:
- **Information theory / signal processing:** a VAE is a lossy compressor; the KL term is the **rate** and the reconstruction error is the **distortion**, and β moves along the rate-distortion curve.
- **Generation quality:** measured with a **Fréchet distance computed in the feature space of a Fashion-MNIST classifier** (an FID-like metric, see below), plus a strong control: an autoencoder whose latent space is fitted *after* training with a Gaussian.
- **The latent space itself:** 2-D latent spaces, interpolations, active units, and latent traversals.

**What's inside**
1. Background: autoencoders, the ELBO, the reparameterization trick, β-VAE, rate-distortion
2. Data and a classifier used for evaluation
3. Models (convolutional encoder / decoder)
4. Seven experiments (A to G)
5. Results: reconstruction, generation, rate-distortion, latent space, interpolation, traversals
6. Analysis, limitations, next steps

> Papers: *Auto-Encoding Variational Bayes* (Kingma and Welling, 2014) · *β-VAE* (Higgins et al., 2017) · *From Variational to Deterministic Autoencoders* (Ghosh et al., 2020) · *GANs Trained by a Two Time-Scale Update Rule* (Heusel et al., 2017, the FID metric)

> ⚙️ **Kaggle setup:** Settings → Accelerator → **GPU** and **Internet → On** (the dataset is downloaded automatically). No "Add Input" is needed. Expected runtime: roughly 15 to 25 minutes.

## 1. Background

### Autoencoder
An autoencoder learns two functions: an **encoder** $z = f(x)$ that compresses an image (784 pixels) into a small **latent code** $z$ (here 16 or 2 numbers), and a **decoder** $\hat{x} = g(z)$ that reconstructs it. It is trained to minimize the reconstruction error. The bottleneck forces it to keep only the most important information. (A linear autoencoder with squared error learns the same subspace as PCA; the nonlinear one can do much better.)

**The problem for generation.** The decoder is only trained on the codes of real images. These codes occupy irregular regions of the latent space, with "holes" in between. If you sample a random $z$ and decode it, you usually land in a hole and get a meaningless image.

### Variational autoencoder
A VAE makes the latent space regular. It is a probabilistic model $p(x, z) = p(z)\,p_\theta(x \mid z)$ with a simple prior $p(z) = \mathcal{N}(0, I)$. The encoder outputs a **distribution** $q_\phi(z \mid x) = \mathcal{N}(\mu(x), \mathrm{diag}\,\sigma^2(x))$. Training maximizes the **evidence lower bound (ELBO)**:

$$\log p(x) \;\ge\; \underbrace{\mathbb{E}_{q_\phi(z \mid x)}\big[\log p_\theta(x \mid z)\big]}_{\text{reconstruction}} \;-\; \underbrace{D_{KL}\big(q_\phi(z \mid x)\,\|\,p(z)\big)}_{\text{regularizer}}$$

- **Reparameterization trick:** $z = \mu + \sigma \odot \epsilon$ with $\epsilon \sim \mathcal{N}(0, I)$, so that the sampling step is differentiable with respect to $\mu$ and $\sigma$.
- **Closed-form KL** for diagonal Gaussians: $D_{KL} = \tfrac{1}{2}\sum_j \big(\mu_j^2 + \sigma_j^2 - \log \sigma_j^2 - 1\big)$.
- The decoder outputs Bernoulli probabilities per pixel (reconstruction = binary cross-entropy, summed over pixels). Fashion-MNIST pixels are continuous in [0, 1], so this is not an exact likelihood and the ELBO values are not comparable with the literature.

### β-VAE and rate-distortion
The β-VAE minimizes $\;\text{reconstruction} + \beta \cdot D_{KL}$. In the language of **rate-distortion theory**, the KL term is the **rate** (the information, in nats, that the code carries about the image) and the reconstruction error is the **distortion**. β is the Lagrange multiplier that selects a point on the rate-distortion curve: a small β allows a high rate and low distortion (autoencoder-like), a large β forces a low rate and high distortion. With a very large β the encoder may ignore the input in some dimensions entirely (**posterior collapse**: those dimensions have KL ≈ 0). A dimension is called **active** if the variance of its posterior mean over the data exceeds 0.01 (Burda et al., 2016).

### Evaluating generation without FID
The standard FID uses an Inception network trained on ImageNet, which is a poor fit for 28×28 clothing images and needs a large download. I use the same idea with a feature extractor suited to the data: a small CNN **trained on Fashion-MNIST**. I compute the **Fréchet distance** between two Gaussians fitted to the 128-dimensional features of real test images and of generated images:

$$d^2 = \|\mu_1 - \mu_2\|^2 + \mathrm{Tr}\big(\Sigma_1 + \Sigma_2 - 2(\Sigma_1 \Sigma_2)^{1/2}\big)$$

Lower is better. I call it **FD-CNN**; it is *not* comparable with published FID numbers. For scale I also report the distance between two sets of real images (the floor), the classifier's **confidence** on generated images (low confidence suggests ambiguous or garbage images), and the **entropy of the predicted class histogram** (diversity; the maximum is ln 10 = 2.30).

### A fair control for the autoencoder
Sampling an autoencoder from $\mathcal{N}(0, I)$ is unfair, because nothing made its codes look like $\mathcal{N}(0, I)$. I therefore also sample every model from a **Gaussian fitted to the codes of the training images** (mean and covariance), following Ghosh et al. (2020). This tells how much of the VAE's advantage is just a matter of knowing where the codes are.

### Experiment design

| ID | Model | Latent dim | β |
|----|-------|-----------|---|
| A | autoencoder | 16 | – |
| B | VAE | 16 | 1 |
| C | β-VAE | 16 | 0.1 |
| D | β-VAE | 16 | 4 |
| E | β-VAE | 16 | 10 |
| F | autoencoder | 2 | – |
| G | VAE | 2 | 1 |

Same convolutional architecture, optimizer, 30 epochs, batch size 128 and seed for all. Models are evaluated after the last epoch (no early stopping, no selection on the test set).

In [ ]:
import os, copy, math, time, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from scipy import linalg

SEED = 42
EPOCHS = 30
BATCH_SIZE = 128
LR = 1e-3
N_SAMPLES = 10000       # generated images per model for the evaluation
ACTIVE_THRESHOLD = 0.01

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(exist_ok=True)

def save_fig(name):
    plt.savefig(OUT / name, dpi=150, bbox_inches="tight")

CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
print("PyTorch:", torch.__version__, "| Device:", DEVICE)

## 2. Data

**Fashion-MNIST** (Xiao et al., 2017): 60,000 training and 10,000 test grayscale images of 28×28 pixels, 10 clothing classes. It is the same size as MNIST but much harder to model. I hold out 5,000 training images for monitoring the loss curves; the rest (55,000) is used for training. All the images are kept on the GPU as tensors in [0, 1].

In [ ]:
def load_fashion_mnist():
    try:
        from torchvision import datasets
        tr = datasets.FashionMNIST("./data", train=True, download=True)
        te = datasets.FashionMNIST("./data", train=False, download=True)
        return tr.data.numpy(), tr.targets.numpy(), te.data.numpy(), te.targets.numpy()
    except Exception as e:
        print("torchvision download failed:", repr(e)[:150], "-> trying Hugging Face datasets")
    from datasets import load_dataset
    try:
        ds = load_dataset("zalando-datasets/fashion_mnist")
    except Exception:
        ds = load_dataset("fashion_mnist")
    arr = lambda split: (np.stack([np.array(im) for im in ds[split]["image"]]), np.array(ds[split]["label"]))
    (xtr, ytr), (xte, yte) = arr("train"), arr("test")
    return xtr, ytr, xte, yte

xtr_all, ytr_all, xte_np, yte_np = load_fashion_mnist()
perm = np.random.RandomState(SEED).permutation(len(xtr_all))
va_idx, tr_idx = perm[:5000], perm[5000:]

to_img = lambda a: torch.as_tensor(a, dtype=torch.float32, device=DEVICE).div(255.0).unsqueeze(1)
Xtr, Xva, Xte = to_img(xtr_all[tr_idx]), to_img(xtr_all[va_idx]), to_img(xte_np)
Ytr = torch.as_tensor(ytr_all[tr_idx], device=DEVICE); Yva = torch.as_tensor(ytr_all[va_idx], device=DEVICE)
Yte = torch.as_tensor(yte_np, device=DEVICE)
print("train", tuple(Xtr.shape), "| val", tuple(Xva.shape), "| test", tuple(Xte.shape), "| pixel range", Xtr.min().item(), Xtr.max().item())

def show_grid(ax, imgs, ncols):
    """imgs: (N, 1, 28, 28) tensor -> one tiled picture"""
    imgs = imgs.detach().cpu().numpy()[:, 0]
    n = len(imgs); nrows = math.ceil(n / ncols)
    canvas = np.ones((nrows * 28, ncols * 28))
    for k, im in enumerate(imgs):
        r, c = divmod(k, ncols); canvas[r * 28:(r + 1) * 28, c * 28:(c + 1) * 28] = im
    ax.imshow(canvas, cmap="gray_r", vmin=0, vmax=1); ax.axis("off")

fig, ax = plt.subplots(figsize=(12, 2.6))
show_grid(ax, Xte[:32], 16)
plt.title("Fashion-MNIST test images")
save_fig("data_samples.png")
plt.show()

## 3. A Classifier for Evaluation (FD-CNN)

A small CNN is trained on the training images only. Its 128-dimensional penultimate layer serves as the feature space for the Fréchet distance. It is also used to measure the confidence on generated images and how much class information survives a reconstruction. To calibrate the scale of the metric, I report the distance between **two sets of real images** (a floor that no generator can beat on average).

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(1, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
                                  nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten())
        self.fc = nn.Sequential(nn.Linear(64 * 5 * 5, 128), nn.ReLU())
        self.drop = nn.Dropout(0.3)
        self.out = nn.Linear(128, 10)
    def features(self, x):
        return self.fc(self.conv(x))
    def forward(self, x):
        return self.out(self.drop(self.features(x)))

torch.manual_seed(SEED)
clf = Classifier().to(DEVICE)
opt = optim.Adam(clf.parameters(), lr=1e-3)
for ep in range(6):
    clf.train()
    order = torch.randperm(len(Xtr), device=DEVICE)
    for i in range(0, len(order), 128):
        j = order[i:i + 128]
        loss = F.cross_entropy(clf(Xtr[j]), Ytr[j])
        opt.zero_grad(); loss.backward(); opt.step()
clf.eval()

@torch.no_grad()
def clf_features_logits(X, bs=1000):
    fs, ls = [], []
    for i in range(0, len(X), bs):
        f = clf.features(X[i:i + bs]); fs.append(f); ls.append(clf.out(f))
    return torch.cat(fs).cpu().numpy().astype(np.float64), torch.cat(ls)

feat_te, logits_te = clf_features_logits(Xte)
clf_acc = (logits_te.argmax(1) == Yte).float().mean().item()
print(f"classifier test accuracy: {clf_acc:.4f}")

def stats(f):
    return f.mean(0), np.cov(f, rowvar=False)

def frechet(mu1, s1, mu2, s2, eps=1e-6):
    diff = mu1 - mu2
    covmean = linalg.sqrtm(s1 @ s2)
    if not np.isfinite(covmean).all():
        off = np.eye(s1.shape[0]) * eps
        covmean = linalg.sqrtm((s1 + off) @ (s2 + off))
    covmean = np.real(covmean)
    return float(diff @ diff + np.trace(s1) + np.trace(s2) - 2 * np.trace(covmean))

mu_te, cov_te = stats(feat_te)
feat_tr_ref, _ = clf_features_logits(Xtr[:10000])
FD_FLOOR = frechet(*stats(feat_tr_ref), mu_te, cov_te)
conf_real = torch.softmax(logits_te, 1).max(1).values.mean().item()
print(f"FD-CNN between 10,000 real training images and the real test images (floor): {FD_FLOOR:.3f}")
print(f"mean classifier confidence on real test images: {conf_real:.3f}")

## 4. Models

Both models share the same convolutional encoder and decoder:

| Part | Layers |
|------|--------|
| Encoder | Conv 1→32 (4×4, stride 2) → ReLU → Conv 32→64 (4×4, stride 2) → ReLU → flatten (64×7×7 = 3,136) → linear → latent |
| Decoder | linear → 3,136 → ReLU → reshape → ConvTranspose 64→32 (4×4, stride 2) → ReLU → ConvTranspose 32→1 (4×4, stride 2) → logits |

- **AutoEncoder:** one linear layer produces $z$.
- **VAE:** two linear layers produce $\mu$ and $\log \sigma^2$; $z = \mu + \sigma \epsilon$ (reparameterization trick).

In [ ]:
class Encoder(nn.Module):
    def __init__(self, z_dim, variational):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 32, 4, 2, 1), nn.ReLU(), nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(), nn.Flatten())
        self.mu = nn.Linear(64 * 7 * 7, z_dim)
        self.logvar = nn.Linear(64 * 7 * 7, z_dim) if variational else None
    def forward(self, x):
        h = self.net(x)
        return self.mu(h), (self.logvar(h).clamp(-10, 10) if self.logvar is not None else None)

class Decoder(nn.Module):
    def __init__(self, z_dim):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(z_dim, 64 * 7 * 7), nn.ReLU())
        self.net = nn.Sequential(nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(), nn.ConvTranspose2d(32, 1, 4, 2, 1))
    def forward(self, z):
        return self.net(self.fc(z).view(-1, 64, 7, 7))          # logits; sigmoid gives pixel probabilities

class AutoEncoder(nn.Module):
    variational = False
    def __init__(self, z_dim):
        super().__init__()
        self.z_dim, self.enc, self.dec = z_dim, Encoder(z_dim, False), Decoder(z_dim)
    def code(self, x):                                          # deterministic code
        return self.enc(x)[0]
    def forward(self, x):
        z = self.code(x)
        return self.dec(z), z

class VAE(nn.Module):
    variational = True
    def __init__(self, z_dim):
        super().__init__()
        self.z_dim, self.enc, self.dec = z_dim, Encoder(z_dim, True), Decoder(z_dim)
    def code(self, x):                                          # posterior mean
        return self.enc(x)[0]
    def forward(self, x):
        mu, logvar = self.enc(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)   # reparameterization trick
        return self.dec(z), mu, logvar

def kl_per_image(mu, logvar):
    return 0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum(1)

def rec_per_image(logits, x):
    return F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum((1, 2, 3))

def count_params(m):
    return sum(p.numel() for p in m.parameters())

for name, m in [("AutoEncoder z=16", AutoEncoder(16)), ("VAE z=16", VAE(16)), ("AutoEncoder z=2", AutoEncoder(2)), ("VAE z=2", VAE(2))]:
    print(f"{name:18s} {count_params(m):,} parameters")
x = Xtr[:4].cpu()
print("VAE forward shapes:", [tuple(t.shape) for t in VAE(16)(x)], "| AE forward shapes:", [tuple(t.shape) for t in AutoEncoder(16)(x)])

## 5. Training and Evaluation

**Loss.** Autoencoder: reconstruction (binary cross-entropy summed over pixels). VAE: reconstruction + β · KL. All values are **per image** (in nats), averaged over the batch. Adam (lr 1e-3), batch size 128, 30 epochs, no early stopping.

**Evaluation on the test set, after the last epoch:**
- **Reconstruction:** BCE per image, and the mean squared error per pixel of the reconstruction from the posterior mean (VAE) or the code (autoencoder); the accuracy of the classifier on the reconstructions (how much class information survives).
- **Rate:** KL per image (nats) of the VAE encoder, and the number of **active units**.
- **Negative ELBO** (reconstruction + KL, i.e. β = 1) for the VAEs.
- **Generation:** 10,000 images decoded from (a) $z \sim \mathcal{N}(0, I)$ and (b) a Gaussian fitted to the training codes. For each: **FD-CNN** against the real test images, the classifier's mean confidence, and the entropy of the predicted-class histogram.

In [ ]:
def run_epoch(model, X, beta, optimizer=None):
    train = optimizer is not None
    model.train(train)
    order = torch.randperm(len(X), device=DEVICE) if train else torch.arange(len(X), device=DEVICE)
    tot_rec, tot_kl, n = 0.0, 0.0, 0
    with torch.set_grad_enabled(train):
        for i in range(0, len(order), BATCH_SIZE):
            x = X[order[i:i + BATCH_SIZE]]
            if model.variational:
                logits, mu, logvar = model(x)
                rec, kl = rec_per_image(logits, x).mean(), kl_per_image(mu, logvar).mean()
                loss = rec + beta * kl
            else:
                logits, _ = model(x)
                rec = rec_per_image(logits, x).mean(); kl = torch.zeros((), device=DEVICE); loss = rec
            if train:
                optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
            tot_rec += rec.item() * len(x); tot_kl += kl.item() * len(x); n += len(x)
    return tot_rec / n, tot_kl / n

@torch.no_grad()
def codes_of(model, X, bs=1000):
    model.eval()
    return torch.cat([model.code(X[i:i + bs]) for i in range(0, len(X), bs)])

@torch.no_grad()
def decode_np_z(model, z_np, bs=1000):
    model.eval()
    out = []
    for i in range(0, len(z_np), bs):
        z = torch.as_tensor(z_np[i:i + bs], dtype=torch.float32, device=DEVICE)
        out.append(torch.sigmoid(model.dec(z)))
    return torch.cat(out)

def generation_metrics(model, z_np):
    imgs = decode_np_z(model, z_np)
    f, logits = clf_features_logits(imgs)
    p = torch.softmax(logits, 1)
    hist = torch.bincount(p.argmax(1), minlength=10).float(); hist = hist / hist.sum()
    entropy = float(-(hist[hist > 0] * hist[hist > 0].log()).sum())
    return dict(fd=frechet(*stats(f), mu_te, cov_te), conf=p.max(1).values.mean().item(), entropy=entropy), imgs[:64].cpu()

@torch.no_grad()
def evaluate(model, beta):
    model.eval()
    rec_sum, kl_sum, mse_sum, n = 0.0, 0.0, 0.0, 0
    mus, kls, recon_imgs = [], [], []
    for i in range(0, len(Xte), 1000):
        x = Xte[i:i + 1000]
        if model.variational:
            logits, mu, logvar = model(x)
            kl_sum += kl_per_image(mu, logvar).sum().item(); mus.append(mu)
            kls.append(0.5 * (mu ** 2 + logvar.exp() - logvar - 1))
            recon = torch.sigmoid(model.dec(mu))
        else:
            logits, z = model(x); recon = torch.sigmoid(logits)
        rec_sum += rec_per_image(logits, x).sum().item(); n += len(x)
        mse_sum += ((recon - x) ** 2).mean((1, 2, 3)).sum().item(); recon_imgs.append(recon)
    recon_imgs = torch.cat(recon_imgs)
    r = dict(rec_bce=rec_sum / n, mse=mse_sum / n)
    r["recon_acc"] = (clf_features_logits(recon_imgs)[1].argmax(1) == Yte).float().mean().item()
    if model.variational:
        mu_all = torch.cat(mus)
        r["kl"] = kl_sum / n; r["neg_elbo"] = r["rec_bce"] + r["kl"]
        r["active"] = int((mu_all.var(0) > ACTIVE_THRESHOLD).sum().item())
        r["kl_per_dim"] = torch.cat(kls).mean(0).cpu().numpy()
    # generation: prior N(0, I) and a Gaussian fitted to the training codes
    rng = np.random.RandomState(SEED)
    g_prior, imgs_prior = generation_metrics(model, rng.randn(N_SAMPLES, model.z_dim))
    ctr = codes_of(model, Xtr).cpu().numpy().astype(np.float64)
    cov = np.atleast_2d(np.cov(ctr, rowvar=False)) + 1e-6 * np.eye(model.z_dim)
    g_fit, imgs_fit = generation_metrics(model, rng.multivariate_normal(ctr.mean(0), cov, N_SAMPLES))
    r.update(fd_prior=g_prior["fd"], conf_prior=g_prior["conf"], ent_prior=g_prior["entropy"],
             fd_fit=g_fit["fd"], conf_fit=g_fit["conf"], ent_fit=g_fit["entropy"])
    r["samples_prior"], r["samples_fit"] = imgs_prior, imgs_fit
    return r

def train_experiment(name, kind, z_dim, beta):
    print(f"\n=== {name} | {kind} | z={z_dim} | beta={beta} ===")
    torch.manual_seed(SEED)
    model = (VAE(z_dim) if kind == "vae" else AutoEncoder(z_dim)).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    hist = {"train_rec": [], "train_kl": [], "val_rec": [], "val_kl": []}
    t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        tr_rec, tr_kl = run_epoch(model, Xtr, beta, optimizer)
        va_rec, va_kl = run_epoch(model, Xva, beta)
        for k, v in zip(hist, [tr_rec, tr_kl, va_rec, va_kl]):
            hist[k].append(v)
        if ep == 1 or ep % 5 == 0:
            print(f"Epoch {ep:02d}/{EPOCHS} | train rec {tr_rec:.2f} KL {tr_kl:.2f} | val rec {va_rec:.2f} KL {va_kl:.2f} | {time.time() - t0:.0f}s")
    elapsed = time.time() - t0
    r = evaluate(model, beta)
    print(f"-> test: rec BCE {r['rec_bce']:.2f} | MSE {r['mse']:.4f}" +
          (f" | KL {r['kl']:.2f} | active units {r['active']}/{z_dim}" if kind == "vae" else "") +
          f" | FD-CNN prior {r['fd_prior']:.2f}, fitted {r['fd_fit']:.2f} | recon acc {r['recon_acc']:.3f}")
    r.update(model=model, hist=hist, kind=kind, z_dim=z_dim, beta=beta, params=count_params(model), time=elapsed)
    return r

results = {}

### Experiment A: AE (z=16)

In [ ]:
results["A: AE (z=16)"] = train_experiment("A: AE (z=16)", "ae", 16, 0.0)

### Experiment B: VAE (z=16, beta=1)

In [ ]:
results["B: VAE (z=16, beta=1)"] = train_experiment("B: VAE (z=16, beta=1)", "vae", 16, 1.0)

### Experiment C: VAE (z=16, beta=0.1)

In [ ]:
results["C: VAE (z=16, beta=0.1)"] = train_experiment("C: VAE (z=16, beta=0.1)", "vae", 16, 0.1)

### Experiment D: VAE (z=16, beta=4)

In [ ]:
results["D: VAE (z=16, beta=4)"] = train_experiment("D: VAE (z=16, beta=4)", "vae", 16, 4.0)

### Experiment E: VAE (z=16, beta=10)

In [ ]:
results["E: VAE (z=16, beta=10)"] = train_experiment("E: VAE (z=16, beta=10)", "vae", 16, 10.0)

### Experiment F: AE (z=2)

In [ ]:
results["F: AE (z=2)"] = train_experiment("F: AE (z=2)", "ae", 2, 0.0)

### Experiment G: VAE (z=2, beta=1)

In [ ]:
results["G: VAE (z=2, beta=1)"] = train_experiment("G: VAE (z=2, beta=1)", "vae", 2, 1.0)

## 6. Results

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "Config": name,
        "z": r["z_dim"],
        "Params": f"{r['params']:,}",
        "Rec BCE (nats/img)": round(r["rec_bce"], 2),
        "Rec MSE (per pixel)": round(r["mse"], 4),
        "KL (nats/img)": round(r["kl"], 2) if r["kind"] == "vae" else None,
        "-ELBO (beta=1)": round(r["neg_elbo"], 2) if r["kind"] == "vae" else None,
        "Active units": f"{r['active']}/{r['z_dim']}" if r["kind"] == "vae" else None,
        "Recon accuracy (%)": round(r["recon_acc"] * 100, 1),
        "FD-CNN (N(0,I))": round(r["fd_prior"], 2),
        "FD-CNN (fitted Gaussian)": round(r["fd_fit"], 2),
        "Confidence (N(0,I))": round(r["conf_prior"], 3),
        "Class entropy (N(0,I))": round(r["ent_prior"], 2),
        "Time (s)": round(r["time"]),
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "results_summary.csv", index=False)
print(f"References: FD-CNN floor between two sets of real images = {FD_FLOOR:.2f} | classifier accuracy on real test images = "
      f"{clf_acc * 100:.1f}% | mean confidence on real images = {conf_real:.3f} | maximum class entropy = {math.log(10):.2f}")
summary

In [ ]:
# Training curves (validation set): reconstruction and KL per image
names = list(results)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.3))
for n in names:
    h = results[n]["hist"]; ep = np.arange(1, EPOCHS + 1)
    ax[0].plot(ep, h["val_rec"], label=n)
    if results[n]["kind"] == "vae":
        ax[1].plot(ep, h["val_kl"], label=n)
ax[0].set_title("Validation reconstruction loss (BCE per image)"); ax[1].set_title("Validation KL per image (VAEs)")
ax[0].set_ylim(top=min(max(results[n]["hist"]["val_rec"][0] for n in names), 400))
for a in ax:
    a.set_xlabel("epoch"); a.grid(alpha=0.3)
ax[0].legend(fontsize=7); ax[1].legend(fontsize=7)
plt.tight_layout()
save_fig("training_curves_all.png")
plt.show()

### Rate-distortion: what does β do?

For the four 16-dimensional VAEs (β = 0.1, 1, 4, 10), the figure shows (left) the **rate**, i.e. the KL per image, as a function of β; (middle) the **rate-distortion curve** (distortion = reconstruction MSE per pixel); (right) the generation quality (FD-CNN) as a function of β, with the 16-dimensional autoencoder as a reference line.

In [ ]:
vae16 = [n for n in names if results[n]["kind"] == "vae" and results[n]["z_dim"] == 16]
vae16.sort(key=lambda n: results[n]["beta"])
betas = [results[n]["beta"] for n in vae16]
ae16 = next(n for n in names if results[n]["kind"] == "ae" and results[n]["z_dim"] == 16)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
ax[0].plot(betas, [results[n]["kl"] for n in vae16], marker="o"); ax[0].set_xscale("log")
ax[0].set_xlabel("beta"); ax[0].set_ylabel("rate: KL (nats per image)"); ax[0].set_title("Rate vs beta")

kls, mses = [results[n]["kl"] for n in vae16], [results[n]["mse"] for n in vae16]
ax[1].plot(kls, mses, marker="o")
for b, k, m in zip(betas, kls, mses):
    ax[1].annotate(f"beta={b}", (k, m), textcoords="offset points", xytext=(6, 6), fontsize=9)
ax[1].axhline(results[ae16]["mse"], color="gray", ls="--", label="autoencoder (z=16)")
ax[1].set_xlabel("rate: KL (nats per image)"); ax[1].set_ylabel("distortion: reconstruction MSE per pixel")
ax[1].set_title("Rate-distortion curve"); ax[1].legend()

ax[2].plot(betas, [results[n]["fd_prior"] for n in vae16], marker="o", label="VAE, z ~ N(0, I)")
ax[2].plot(betas, [results[n]["fd_fit"] for n in vae16], marker="s", label="VAE, fitted Gaussian")
ax[2].axhline(results[ae16]["fd_prior"], color="tab:red", ls="--", label="AE, z ~ N(0, I)")
ax[2].axhline(results[ae16]["fd_fit"], color="tab:green", ls="--", label="AE, fitted Gaussian")
ax[2].axhline(FD_FLOOR, color="k", ls=":", label="real vs real (floor)")
ax[2].set_xscale("log"); ax[2].set_yscale("log"); ax[2].set_xlabel("beta"); ax[2].set_ylabel("FD-CNN (lower is better)")
ax[2].set_title("Generation quality vs beta"); ax[2].legend(fontsize=7)
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
save_fig("rate_distortion.png")
plt.show()

### Reconstructions and generated samples

Top: test images and their reconstructions by the 16-dimensional models. Bottom: 16 random samples from each model, decoded from $z \sim \mathcal{N}(0, I)$, plus the autoencoder sampled from a Gaussian fitted to its training codes.

In [ ]:
with torch.no_grad():
    originals = Xte[:16]
    recon_rows = [("original", originals)]
    for n in [ae16] + vae16:
        m = results[n]["model"].eval()
        r_ = torch.sigmoid(m(originals)[0]) if not m.variational else torch.sigmoid(m.dec(m.code(originals)))
        recon_rows.append((n, r_))

fig, axes = plt.subplots(len(recon_rows), 1, figsize=(12, 1.15 * len(recon_rows)))
for ax, (label, imgs) in zip(axes, recon_rows):
    show_grid(ax, imgs, 16); ax.set_title(label, fontsize=8, loc="left")
plt.tight_layout()
save_fig("reconstructions.png")
plt.show()

sample_rows = [(ae16 + "  | z ~ N(0, I)", results[ae16]["samples_prior"][:16]),
               (ae16 + "  | fitted Gaussian", results[ae16]["samples_fit"][:16])]
for n in vae16:
    sample_rows.append((n + "  | z ~ N(0, I)", results[n]["samples_prior"][:16]))
fig, axes = plt.subplots(len(sample_rows), 1, figsize=(12, 1.15 * len(sample_rows)))
for ax, (label, imgs) in zip(axes, sample_rows):
    show_grid(ax, imgs, 16); ax.set_title(label, fontsize=8, loc="left")
plt.tight_layout()
save_fig("samples.png")
plt.show()

### The 2-D latent spaces (F: autoencoder, G: VAE)

With only 2 latent dimensions the codes can be plotted directly. The scatter plots show the codes of 5,000 test images colored by class; the grids decode a regular 15×15 grid of latent points. A good generative latent space has **no holes**: every point decodes to a plausible garment.

In [ ]:
f_name, g_name = next(n for n in names if n.startswith("F:")), next(n for n in names if n.startswith("G:"))
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, n in zip(ax, [f_name, g_name]):
    z = codes_of(results[n]["model"], Xte[:5000]).cpu().numpy()
    sc = a.scatter(z[:, 0], z[:, 1], c=yte_np[:5000], cmap="tab10", s=5, alpha=0.7)
    a.set_title(f"{n}: codes of 5,000 test images"); a.set_xlabel("z1"); a.set_ylabel("z2"); a.grid(alpha=0.3)
handles = [plt.Line2D([], [], marker="o", ls="", color=plt.cm.tab10(i / 9), label=c) for i, c in enumerate(CLASSES)]
ax[1].legend(handles=handles, fontsize=7, loc="upper right", ncol=2)
plt.tight_layout()
save_fig("latent_2d.png")
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
for a, n in zip(ax, [f_name, g_name]):
    zc = codes_of(results[n]["model"], Xtr).cpu().numpy()
    if results[n]["kind"] == "vae":
        lo, hi = np.array([-3.0, -3.0]), np.array([3.0, 3.0])
    else:
        lo, hi = np.percentile(zc, 1, axis=0), np.percentile(zc, 99, axis=0)
    g1, g2 = np.linspace(lo[0], hi[0], 15), np.linspace(hi[1], lo[1], 15)
    grid = np.array([[x1, x2] for x2 in g2 for x1 in g1])
    imgs = decode_np_z(results[n]["model"], grid)
    show_grid(a, imgs, 15)
    a.set_title(f"{n}: decoded grid over z1 in [{lo[0]:.1f}, {hi[0]:.1f}], z2 in [{lo[1]:.1f}, {hi[1]:.1f}]", fontsize=9)
plt.tight_layout()
save_fig("manifold_2d.png")
plt.show()

### Interpolation in latent space

I encode two test images of different classes, interpolate linearly between their codes (the posterior means for the VAE) in 9 steps, and decode. A smooth, plausible path means the latent space is well organized.

In [ ]:
first_of = lambda c: int((Yte == c).nonzero()[0])
pairs = [(first_of(0), first_of(7)), (first_of(3), first_of(9)), (first_of(8), first_of(1))]   # T-shirt->Sneaker, Dress->Ankle boot, Bag->Trouser
models_i = [ae16, "B: VAE (z=16, beta=1)"]
fig, axes = plt.subplots(len(pairs) * 2, 1, figsize=(10, 1.3 * len(pairs) * 2))
row = 0
with torch.no_grad():
    for (a, b) in pairs:
        for n in models_i:
            m = results[n]["model"].eval()
            za, zb = m.code(Xte[a:a + 1]), m.code(Xte[b:b + 1])
            ts = torch.linspace(0, 1, 9, device=DEVICE).view(-1, 1)
            imgs = torch.sigmoid(m.dec((1 - ts) * za + ts * zb))
            show_grid(axes[row], imgs, 9); axes[row].set_title(n, fontsize=8, loc="left"); row += 1
plt.tight_layout()
save_fig("interpolation.png")
plt.show()

### What does each latent dimension encode? (per-dimension KL and traversals)

The KL of each latent dimension shows how much information it carries; dimensions with KL close to 0 are unused (**posterior collapse**). For the most informative dimensions of a VAE, a **traversal** decodes the code with that one dimension swept from −3 to +3 and all the others set to 0, which shows which visual factor the dimension controls.

In [ ]:
vae_show = [n for n in vae16 if results[n]["beta"] in (1.0, 4.0, 10.0)]
fig, ax = plt.subplots(1, len(vae_show), figsize=(5 * len(vae_show), 3.6), sharey=True)
for a, n in zip(np.atleast_1d(ax), vae_show):
    kd = results[n]["kl_per_dim"]
    a.bar(range(len(kd)), kd, color=["tab:blue" if v > 0.05 else "lightgray" for v in kd])
    a.set_title(f"{n}\nactive units: {results[n]['active']}/16", fontsize=9); a.set_xlabel("latent dimension")
np.atleast_1d(ax)[0].set_ylabel("average KL (nats)")
plt.tight_layout()
save_fig("per_dim_kl.png")
plt.show()

trav = [n for n in vae16 if results[n]["beta"] in (1.0, 4.0)]
vals = torch.linspace(-3, 3, 9, device=DEVICE)
fig, axes = plt.subplots(5, len(trav), figsize=(6.4 * len(trav), 7.2))
axes = axes.reshape(5, len(trav))
with torch.no_grad():
    for col, n in enumerate(trav):
        top = np.argsort(-results[n]["kl_per_dim"])[:5]
        m = results[n]["model"].eval()
        for row, d in enumerate(top):
            z = torch.zeros(9, 16, device=DEVICE); z[:, d] = vals
            show_grid(axes[row, col], torch.sigmoid(m.dec(z)), 9)
            axes[row, col].set_title(f"{n}: dimension {d} (KL {results[n]['kl_per_dim'][d]:.2f}), from -3 to +3", fontsize=8, loc="left")
plt.tight_layout()
save_fig("traversals.png")
plt.show()

In [ ]:
# Save the best VAE (lowest -ELBO among the beta = 1 models) and the evaluation classifier
b_name = "B: VAE (z=16, beta=1)"
torch.save(results[b_name]["model"].state_dict(), OUT / "vae_z16.pth")
torch.save(clf.state_dict(), OUT / "evaluation_classifier.pth")
print("Saved to", OUT)

## 7. Analysis

### Key findings

**1. Regularization has a price in reconstruction.** At the same latent size (16), the autoencoder reconstructs better than the VAE with β = 1: reconstruction MSE 0.0097 vs 0.0113 (+16.5%), BCE 212.9 vs 223.0 nats (+10 nats), and the classifier recognizes the class of 82.9% of the autoencoder's reconstructions vs 80.1% of the VAE's. With β = 0.1 the VAE is practically as good as the autoencoder (MSE 0.0095, BCE 213.4, 82.6%); the tiny advantage over the autoencoder is within what one seed can produce.

**2. β traces a rate-distortion curve.** The rate (KL per image) falls from 43.6 nats for β = 0.1 (63 bits) to 17.0 nats for β = 1 (24.5 bits), 7.0 for β = 4 (10.1 bits) and 3.8 for β = 10 (5.5 bits), while the distortion (MSE per pixel) rises from 0.0095 to 0.0113, 0.0177 and 0.0243. The curve is convex with a knee near β = 1: going from 3.8 to 17 nats cuts the distortion by 54%, but going from 17 to 44 nats buys only 16% more. The classifier accuracy on reconstructions shows how much class information is kept: 82.6%, 80.1%, 68.2%, 59.3% for β = 0.1, 1, 4, 10. The training curves show that KL and reconstruction loss settle early and stay stable.

**3. Sampling an autoencoder from N(0, I) fails; a VAE works.** The autoencoder's samples from N(0, I) have an FD-CNN of 214.1 and the classifier is very unsure about them (mean confidence 0.469); visually they are texture-like blobs with no garment structure. The VAE with β = 1 reaches an FD-CNN of 68.4 (3.1 times lower), a confidence of 0.729, and a class histogram close to uniform (entropy 2.21 of a maximum of 2.30), and its samples are blurry but recognizable garments (shirts, dresses, shoes, bags, trousers). In 2-D the same pattern holds (FD-CNN 212.0 for the autoencoder vs 132.8 for the VAE; class entropy 1.12 vs 1.87).

**4. The fitted-Gaussian control: most of the VAE's advantage is knowing where the codes are.** If the autoencoder's training codes are fitted with a Gaussian and sampled from it, its FD-CNN drops from 214.1 to **65.8**, which is as good as the VAE (68.4 from the prior, 68.7 fitted). The same happens in 2-D, where the fitted autoencoder (97.0) is even better than the VAE (132.8 from the prior, 110.5 fitted). So at this scale and with this metric, the KL term does not give a better generator; what it gives is that the aggregate posterior is close to N(0, I), so that you can sample from the prior without fitting anything (consistent with Ghosh et al., 2020). All 16-dimensional models end up with an FD-CNN between 64 and 69 when sampled well, far above the real-vs-real floor (about 0.3, read from the figure), so the quality of the samples is probably limited by the blurry decoder and the Bernoulli loss rather than by the latent distribution (a hypothesis). I only looked at 16 samples per model, and the FD-CNN is a crude metric, so I would not claim the autoencoder with a fitted Gaussian is *better* than the VAE; in the samples shown, its images look a bit more broken (ghost shapes, irregular outlines).

**5. β and generation quality.** Sampling from N(0, I), the FD-CNN is 85.6 (β = 0.1), **68.4 (β = 1)**, 83.3 (β = 4), 105.7 (β = 10): a U-shape with the best value at β = 1. A small β gives a good decoder but an aggregate posterior that does not match the prior (holes), while a large β matches the prior but loses detail and diversity (class entropy falls from 2.21 to 1.94 while the classifier confidence rises, a sign of blurry prototypes). Caveat on the "fitted Gaussian" numbers for VAEs: I fitted the Gaussian to the **posterior means**, which ignore the posterior noise, so for large β (where the posterior variance is large) the fitted distribution is too narrow; this explains why it is worse than the prior at β = 4 and 10 (96.2 vs 83.3 and 119.0 vs 105.7). The fitted-Gaussian control is therefore clean for the autoencoder but only approximate for the VAEs.

**6. Posterior collapse appears as β grows.** Active units: 16 of 16 for β = 1 (the per-dimension KL is spread between about 0.2 and 2.5 nats), 10 of 16 for β = 4, and 5 of 16 for β = 10, where almost all the information is carried by two dimensions with about 1.5 nats each. The unused dimensions have a KL of 0: the encoder ignores the input there and outputs the prior. A larger β therefore trades *which* information is kept for a smaller budget.

**7. The 2-D latent spaces.** The 2-D autoencoder and the 2-D VAE reconstruct almost identically (MSE 0.0292 vs 0.0290, class accuracy 56.3% for both): with only two dimensions the model is already rate-limited (KL 6.5 nats, about 9 bits), so the regularizer costs nothing. The scatter plots look alike in topology (shoes on one side, trousers and dresses on the other, shirts, pullovers and coats overlapping in the middle), but the autoencoder's codes spread over a large, unbounded area (z1 from about -5 to 12, z2 from about -2.5 to 13.5, with elongated arms) whereas the VAE's stay in a compact region of about ±3 around the origin. In the decoded grids, the autoencoder gives plausible garments where its data lie, but also regions of faint or fragmented images in the empty parts of its latent space (the upper middle of the grid); the VAE's grid covers almost every class with clear garments and smooth transitions, but its periphery has grainy textures (upper left) and nearly blank images (lower right), which are low-density regions of the prior. So "no holes" is only approximately true.

**8. Interpolation: the VAE was not smoother (unexpected).** For three pairs (T-shirt to sneaker, dress to ankle boot, bag to trouser), the autoencoder morphs gradually from one garment to the other through a blurry hybrid. The VAE tends to stay close to the first garment for the first four steps, then changes quickly through a blurry hybrid at the middle steps, and the last three steps are nearly identical. I expected a smoother path for the VAE; with three examples I cannot say this is general, but a plausible explanation (hypothesis) is that the regularized latent space has compact clusters per garment type, so the decoder output is stable inside a cluster and changes abruptly between clusters. In neither model are the intermediate images plausible garments.

**9. Latent traversals: not cleanly disentangled.** In the β = 1 model, the most informative dimensions mostly change the **category** of the item together with its overall darkness (dimension 1 morphs a dark T-shirt into a pullover and then a shoe; dimension 10 morphs a dark bag-like shape into a shirt and then trousers; dimension 15 changes a long-sleeve top into a hooded one). In the β = 4 model, several dimensions are also category-level, with grainy artifacts at the extremes, while dimension 5 looks closer to an attribute (a dark long-sleeve top becomes a lighter short-sleeve T-shirt, i.e. sleeve length and brightness change together). Brightness changes together with the other factors in most rows. With this small architecture and 30 epochs, I do not see the clean single-attribute factors often claimed for β-VAE; the traversals are also done with all other dimensions set to 0, which may leave the data support.

**10. Cost.** The VAE adds 50,192 parameters (the extra log-variance head, 220,385 vs 170,193) and about 28% training time (41 s vs 32 s for 30 epochs). By construction the β = 1 model has the best negative ELBO (239.9 nats; 257.0, 249.8 and 264.9 for β = 0.1, 4 and 10, which optimize a different objective).

### Limitations
- Single run per configuration (one seed); differences between close values (for example autoencoder 65.8 vs VAE 68.4 with a fitted Gaussian) are not conclusive.
- The FD-CNN uses features of a classifier trained on the same data: it is a useful relative measure here, but not FID, not comparable with published numbers, and blind to details the classifier ignores. The Fréchet distance assumes Gaussian features and is biased with a finite number of samples. The real-vs-real floor value (about 0.3) was read from a figure; the classifier accuracy on real images was printed in the notebook but is not part of my analysis.
- For the VAEs, the "fitted Gaussian" uses posterior means (underdispersed), so those numbers are only approximate (see point 5).
- The decoder is a Bernoulli on continuous pixels, so the ELBO values are not true likelihoods and outputs are blurry; only 28×28 images, a small architecture, 30 epochs and no hyperparameter search.
- The sample grids, interpolations and traversals show a handful of examples; interpretations of what a dimension "means" are subjective.

### Possible improvements
- Fit the Gaussian to samples z ~ q(z|x) (the aggregate posterior) for the VAEs; several seeds; more samples for the visual comparison
- KL annealing or free bits to reduce collapse; a Gaussian decoder with learned variance; a richer prior or a hierarchical VAE; test disentanglement metrics
- Real FID on a larger dataset; precision and recall for generative models; compare with a GAN (Day 10)
- Interpolate along the spherical path or in the pixel-space posterior samples; traversals starting from real images instead of the origin

### Next in this series
**Day 10:** DCGAN for image generation, the last project of the series.

### References
- Kingma, D., Welling, M. (2014). *Auto-Encoding Variational Bayes.* ICLR.
- Higgins, I. et al. (2017). *β-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework.* ICLR.
- Burda, Y., Grosse, R., Salakhutdinov, R. (2016). *Importance Weighted Autoencoders.* ICLR (active units).
- Ghosh, P. et al. (2020). *From Variational to Deterministic Autoencoders.* ICLR.
- Heusel, M. et al. (2017). *GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium.* NeurIPS (FID).
- Xiao, H., Rasul, K., Vollgraf, R. (2017). *Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms.*
- Alemi, A. et al. (2018). *Fixing a Broken ELBO.* ICML (rate-distortion view of VAEs).